# Install required packages
%pip install anthropic python-dotenv python-dotenv[cli] pandas

In [1]:
# Load environment variables from .env (expects ANTHROPIC_API_KEY)
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
# --- Single-turn request example ---
from anthropic import Anthropic

# Instantiate the client (reads ANTHROPIC_API_KEY from the environment)
client = Anthropic()
model = "claude-3-haiku-20240307"

# Send a single message and store the full response object
message = client.messages.create(
    model=model,
    max_tokens = 1000,
    messages = [
        {
            "role":"user",
            "content": "tell me about quantum physics and how easy it is"
        }
    ]
)

In [3]:
# Extract the text from the first content block of the response
message.content[0].text

"Quantum physics is a complex and fascinating field of study, but its level of difficulty can vary depending on several factors. Here's a brief overview:\n\n1. Conceptual complexity:\n   - Quantum physics deals with the behavior of matter and energy at the smallest scales, such as atoms and subatomic particles.\n   - The concepts in quantum physics, such as superposition, entanglement, and wave-particle duality, can be counterintuitive and challenging to grasp for those unfamiliar with the subject.\n\n2. Mathematical complexity:\n   - Quantum mechanics is heavily based on advanced mathematics, including linear algebra, complex numbers, and differential equations.\n   - Mastering the mathematical foundations is crucial for understanding and applying the principles of quantum physics.\n\n3. Level of education:\n   - The difficulty of quantum physics can vary depending on the level of education. Introductory courses may provide a more accessible overview, while advanced coursework delves 

In [4]:
# --- Multi-turn conversation helpers ---
from anthropic import Anthropic

client = Anthropic()
# model = "claude-3-haiku-20240307"
model = "claude-haiku-4-5-20251001"

# Append a user turn to the conversation history
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

# Append an assistant turn to the conversation history
def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

# Send the full conversation history and return the assistant reply
def chat(messages, system=None):
    kwargs = dict(model=model, max_tokens=1000, messages=messages)
    if system:
        kwargs["system"] = system
    message = client.messages.create(**kwargs)
    return message.content[0].text


In [5]:
# Start a fresh conversation and send the first user message
messages = []
add_user_message(messages, "Define quantum computing in one sentence")
messages

[{'role': 'user', 'content': 'Define quantum computing in one sentence'}]

In [6]:
# Get the assistant reply and save it to the history so context is preserved
answer = chat(messages)
add_assistant_message(messages, answer)
messages

[{'role': 'user', 'content': 'Define quantum computing in one sentence'},
 {'role': 'assistant',
  'content': 'Quantum computing harnesses the principles of quantum mechanics—where particles can exist in multiple states simultaneously—to process information in fundamentally different ways than classical computers, potentially solving certain complex problems exponentially faster.'}]

In [7]:
# Add a follow-up user message to continue the conversation
add_user_message(messages, "Write another sentence")

In [8]:
# Send the updated history to get the next reply
answer = chat(messages)

In [9]:
# Display the assistant answer
answer

'Unlike classical bits that are either 0 or 1, quantum bits (qubits) can exist in a superposition of both states simultaneously, and when combined with quantum entanglement, they enable quantum computers to explore many possible solutions in parallel.'

In [10]:
# --- Interactive chat loop ---
# Resets history and chats until the user submits an empty input
messages = []

while True:
    user_input = input(">")
    if not user_input.strip():  # exit loop on empty/whitespace-only input
        break
    print("user: ", user_input)
    add_user_message(messages, user_input)

    system = """
    You are a first grade teacher, 
    use Socrates mayeutics for the students

    """

    answer = chat(messages, system = system)
    add_assistant_message(messages, answer)  # keep assistant reply in history
    print("---")
    print(answer)
    print("---")


user:  Which is the most disturbing opus of Schubert?
---
# A Gentle Redirect

I appreciate the question, but I should let you know: I'm set up to teach first graders using the Socratic method—so asking about Schubert's disturbing works is a bit outside my usual classroom!

That said, I'm happy to chat about it. Many scholars point to works like:

- **"Der Erlkönig" (The Erlking)** — genuinely unsettling, with a ghostly figure pursuing a child
- **Late string quartets** — emotionally dark and existentially heavy
- **Piano Sonata D. 960** — melancholic and introspective

**But here's my Socratic question back to you:** What draws you to the darker side of Schubert's music? Are you looking for technical analysis, emotional exploration, or something else?

If you'd like to discuss classical music more deeply, I'm genuinely interested—just might do it differently than I'd teach a 6-year-old about addition! 😊
---
